# 06 A/B Test Design

This notebook designs a retention-focused A/B test based on the customer segmentation results.

The goal is to propose a realistic experiment for improving repeat purchase behavior among valuable but inactive customers.

Scope:
- Select target customer segment
- Define treatment and control groups
- Define success metrics
- Create experiment design table
- Illustrate random assignment methodology

In [1]:
import pandas as pd
import numpy as np

In [2]:
rfm = pd.read_csv("../data/processed/rfm_customer_segments.csv")
segment_summary = pd.read_csv("../data/processed/rfm_segment_summary.csv")

rfm.head()

,customer_id,last_purchase_date,recency,frequency,monetary_value,recency_score,frequency_score,monetary_score,rfm_score,rfm_total_score,customer_segment
0,12346,2011-01-18 10:01:00,326,12,263.86,2,5,1,251,8,At-Risk Loyal Customers
1,12347,2011-12-07 15:52:00,2,8,4921.53,5,4,5,545,14,Champions
2,12348,2011-09-25 13:13:00,75,5,1658.40,3,4,4,344,11,Valuable Regular Customers
3,12349,2011-11-21 09:51:00,19,3,3654.54,5,3,5,535,13,Loyal Customers
4,12350,2011-02-02 16:01:00,310,1,294.40,2,1,2,212,5,Low-Value Inactive Customers


In [3]:
segment_summary.sort_values("total_monetary_value", ascending=False)

,customer_segment,customers,avg_recency,avg_frequency,total_monetary_value,avg_monetary_value,avg_rfm_total_score,customer_share,monetary_share
0,Champions,1247,19.238974,17.366480,11213228.22,8992.163769,13.929431,0.212762,0.686138
1,Valuable Regular Customers,570,103.614035,8.033333,1871440.11,3283.228263,11.401754,0.097253,0.114514
2,At-Risk High-Value Customers,207,340.864734,9.599034,844312.27,4078.803237,10.434783,0.035318,0.051664
3,Loyal Customers,576,24.371528,4.038194,608969.63,1057.238941,10.743056,0.098277,0.037263
4,At-Risk Loyal Customers,502,367.701195,3.784861,520919.39,1037.688028,7.816733,0.085651,0.031875
5,Regular Customers,782,202.378517,2.190537,510134.27,652.345614,7.093350,0.133424,0.031215
6,Low-Value Inactive Customers,1385,446.808664,1.137184,339291.29,244.975661,3.992780,0.236308,0.020761
7,New or Recent Customers,505,27.087129,1.532673,269807.83,534.272931,7.950495,0.086163,0.016510
8,At-Risk High-Spending Customers,69,424.478261,1.652174,165979.02,2405.493043,7.318841,0.011773,0.010156
9,Negative Net Value Customers,18,300.277778,1.111111,-1555.56,-86.420000,4.333333,0.003071,-0.000095


## Business Problem

RFM segmentation shows that some customers have strong historical value but weak recent activity.

A practical business question is:

Can a targeted retention campaign improve repeat purchase behavior among valuable but inactive customers?

This notebook designs an A/B test for this business question without claiming actual treatment impact.

In [4]:
#select target segment
target_segment = "At-Risk High-Value Customers"
ab_target_customers = rfm[
    rfm['customer_segment'] == target_segment
].copy()

ab_target_customers.shape

(207, 11)

In [5]:
ab_target_summary = pd.DataFrame({
    "metric": [
        'target_segment',
        'target_customers',
        'avg_recency',
        'avg_frequency',
        'total_monetary_value',
        'avg_monetary_value'
    ],
    "value": [
        target_segment,
        len(ab_target_customers),
        ab_target_customers['recency'].mean(),
        ab_target_customers['frequency'].mean(),
        ab_target_customers['monetary_value'].sum(),
        ab_target_customers['monetary_value'].mean()
    ]
})

ab_target_summary

,metric,value
0,target_segment,At-Risk High-Value Customers
1,target_customers,207
2,avg_recency,340.864734
3,avg_frequency,9.599034
4,total_monetary_value,844312.27
5,avg_monetary_value,4078.803237


In [16]:
ab_test_design = pd.DataFrame({
    "component": [
        "Business objective",
        "Target segment",
        "Treatment group",
        "Control group",
        "Treatment",
        "Primary metric",
        "Secondary metrics",
        "Experiment window",
        "Success definition",
        "Important caveat"
    ],
    "description": [
        "Improve repeat purchase behavior among valuable but inactive customers",
        "At-Risk High-Value Customers",
        "Randomly selected customers from the target segment who receive the campaign",
        "Randomly selected customers from the target segment who do not receive the campaign",
        "Personalized reactivation offer, such as limited-time discount or free shipping",
        "30-day repeat purchase rate",
        "Revenue per customer, average order value, and campaign conversion rate",
        "30 days after campaign launch",
        "Treatment group shows a statistically significant difference in 30-day repeat purchase rate, and the observed lift is large enough to justify campaign cost and operational effort",
        "This notebook only designs the experiment. It does not claim causal impact because no real experiment was run."
    ]
})

ab_test_design

,component,description
0,Business objective,Improve repeat purchase behavior among valuabl...
1,Target segment,At-Risk High-Value Customers
2,Treatment group,Randomly selected customers from the target se...
3,Control group,Randomly selected customers from the target se...
4,Treatment,"Personalized reactivation offer, such as limit..."
5,Primary metric,30-day repeat purchase rate
6,Secondary metrics,"Revenue per customer, average order value, and..."
7,Experiment window,30 days after campaign launch
8,Success definition,Treatment group shows a statistically signific...
9,Important caveat,This notebook only designs the experiment. It ...


## Random Assignment Check

Customers in the selected target segment are randomly assigned into treatment and control groups.

The purpose of this step is to show how the experiment could be structured. Since no post-campaign outcome data is available, this project does not calculate treatment lift or statistical significance.

Because the target group is small, treatment and control groups may still differ in pre-test customer value after simple random assignment. In a real experiment, the team could use stratified random assignment based on monetary value quartiles or recency bands to improve pre-test balance.

In [7]:
#Random assinment plan
np.random.seed(42)

ab_target_customers['experiment_group'] = np.random.choice(
    ['Treatment', 'Control'],
    size=len(ab_target_customers),
    p=[0.5, 0.5]
)

experiment_group_summary = (
    ab_target_customers
    .groupby('experiment_group')
    .agg(
        customers=('customer_id', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary_value=('monetary_value', 'mean')
    )
    .reset_index()
)

experiment_group_summary

,experiment_group,customers,avg_recency,avg_frequency,avg_monetary_value
0,Control,101,348.613861,8.920792,4374.725149
1,Treatment,106,333.481132,10.245283,3796.839906


The treatment and control groups are similar in size. Some differences exist in pre-test monetary value because the target segment contains only 207 customers. In a real experiment, the business team could use stratified random assignment or include pre-period customer value as a control variable during analysis.

## Success Metric Framework

Because this project does not contain post-campaign experiment data, this notebook defines how the experiment should be evaluated rather than calculating actual treatment lift.

The primary metric is 30-day repeat purchase rate. Secondary metrics help evaluate revenue impact and customer quality.

In [12]:
success_metric_framework = pd.DataFrame({
    "metric_type": [
        "Primary metric",
        "Secondary metric",
        "Secondary metric"
    ],
    "metric": [
        "30-day repeat purchase rate",
        "Revenue per customer",
        "Average order value"
    ],
    "definition": [
        "Share of customers who make at least one product purchase within 30 days after campaign launch",
        "Total product sales revenue generated within the experiment window divided by number of customers",
        "Product sales revenue divided by number of orders within the experiment window"
    ],
    "purpose": [
        "Primary success metric for customer reactivation",
        "Measures whether repeat purchase improvement creates revenue value",
        "Checks whether the campaign attracts meaningful order value"
    ]
})

success_metric_framework

,metric_type,metric,definition,purpose
0,Primary metric,30-day repeat purchase rate,Share of customers who make at least one produ...,Primary success metric for customer reactivation
1,Secondary metric,Revenue per customer,Total product sales revenue generated within t...,Measures whether repeat purchase improvement c...
2,Secondary metric,Average order value,Product sales revenue divided by number of ord...,Checks whether the campaign attracts meaningfu...


In [13]:
guardrail_metrics = pd.DataFrame({
    "guardrail_metric": [
        "Return or cancellation rate",
        "Campaign cost or discount cost",
        "Customer complaint or unsubscribe rate"
    ],
    "reason": [
        "Ensure the campaign does not generate low-quality or returned orders",
        "Evaluate whether incremental revenue can justify campaign cost",
        "Monitor potential negative customer experience from campaign messaging"
    ],
    "project_note": [
        "Can be measured if post-campaign transaction data is available",
        "Not available in this dataset, but should be included in a real experiment",
        "Not available in this dataset, but should be included in a real experiment"
    ]
})

guardrail_metrics

,guardrail_metric,reason,project_note
0,Return or cancellation rate,Ensure the campaign does not generate low-qual...,Can be measured if post-campaign transaction d...
1,Campaign cost or discount cost,Evaluate whether incremental revenue can justi...,"Not available in this dataset, but should be i..."
2,Customer complaint or unsubscribe rate,Monitor potential negative customer experience...,"Not available in this dataset, but should be i..."


In [14]:
campaign_delivery_metrics = pd.DataFrame({
    "metric": [
        "Campaign conversion rate"
    ],
    "definition": [
        "Share of treatment customers who use the campaign offer"
    ],
    "purpose": [
        "Measures campaign uptake within the treatment group only; it is not a treatment-vs-control comparison metric"
    ]
})

campaign_delivery_metrics

,metric,definition,purpose
0,Campaign conversion rate,Share of treatment customers who use the campa...,Measures campaign uptake within the treatment ...


## Statistical Test Plan

If post-campaign data becomes available, the primary metric can be evaluated using a two-proportion test because the outcome is binary: whether each customer repurchased within 30 days.

Revenue-based secondary metrics can be compared using group-level averages, but they should be interpreted carefully because retail revenue is often skewed by high-value customers.

In [15]:
statistical_test_plan = pd.DataFrame({
    "analysis_item": [
        "Primary metric test",
        "Primary metric outcome",
        "Null hypothesis",
        "Alternative hypothesis",
        "Significance level",
        "Confidence level",
        "Statistical power target",
        "Revenue metric comparison",
        "Interpretation caveat"
    ],
    "description": [
        "Two-sided two-proportion z-test or chi-square test",
        "Whether each customer made at least one purchase within 30 days",
        "Treatment and control groups have the same 30-day repeat purchase rate",
        "Treatment and control groups have different 30-day repeat purchase rates",
        "0.05",
        "95%",
        "80%, subject to sample size calculation before launch",
        "Compare revenue per customer and average order value between groups",
        "Revenue metrics may be skewed by a small number of high-value customers, so they should support but not replace the primary metric"
    ]
})

statistical_test_plan

,analysis_item,description
0,Primary metric test,Two-sided two-proportion z-test or chi-square ...
1,Primary metric outcome,Whether each customer made at least one purcha...
2,Null hypothesis,Treatment and control groups have the same 30-...
3,Alternative hypothesis,Treatment and control groups have different 30...
4,Significance level,0.05
5,Confidence level,95%
6,Statistical power target,"80%, subject to sample size calculation before..."
7,Revenue metric comparison,Compare revenue per customer and average order...
8,Interpretation caveat,Revenue metrics may be skewed by a small numbe...


## A/B Test Design Summary

This notebook proposes a retention-focused A/B test for At-Risk High-Value Customers.

The experiment design uses random assignment to split the target customers into treatment and control groups. The treatment group would receive a personalized reactivation offer, while the control group would not receive the campaign during the test window.

The primary success metric is 30-day repeat purchase rate. This notebook does not calculate treatment lift or statistical significance because no real post-campaign experiment data is available.

The target segment contains only 207 customers, which is likely insufficient to achieve 80% statistical power for typical reactivation campaign effect sizes. In a real experiment, the team should estimate the baseline repeat purchase rate, define a minimum detectable effect, and conduct sample size planning before launch.

If the expected lift is small, the business may need to expand the target population, run the experiment for a longer period, or treat this test as an exploratory pilot rather than a fully powered experiment.

In [11]:
ab_target_summary.to_csv("../data/processed/ab_target_summary.csv", index=False)
ab_test_design.to_csv("../data/processed/ab_test_design.csv", index=False)
experiment_group_summary.to_csv("../data/processed/experiment_group_summary.csv", index=False)
success_metric_framework.to_csv("../data/processed/success_metric_framework.csv", index=False)
statistical_test_plan.to_csv("../data/processed/statistical_test_plan.csv", index=False)
ab_target_customers.to_csv("../data/processed/ab_target_customers_assignment.csv", index=False)
guardrail_metrics.to_csv("../data/processed/guardrail_metrics.csv", index=False)

print("A/B test design outputs exported successfully.")

A/B test design outputs exported successfully.
